In [1]:
"""
Bank Loan Risk Intelligence Platform
00_load_and_clean_lendingclub.py

Loads the raw LendingClub 'accepted' loans CSV (downloaded manually per
Documentation/dataset_guide.md), filters it to a tractable, resolved subset,
and writes a cleaned CSV that the rest of the pipeline (WOE/IV notebook, SQL
load, Power BI) consumes.

Before running:
    1. Download accepted_2007_to_2018Q4.csv from Kaggle (wordsforthewise/lending-club)
    2. Place it at Dataset/raw/accepted_2007_to_2018Q4.csv
    3. pip install pandas numpy --break-system-packages

This script deliberately:
    - Keeps only loans with a RESOLVED status (Fully Paid / Charged Off / Default).
      Loans still 'Current' or 'Late' haven't finished playing out, so including
      them as non-defaults would bias the model — a real, common credit-modeling
      pitfall, worth a line in your Key Insight section.
    - Filters to recent-enough vintages (2015+) so the book isn't dominated by
      very old, very small early LendingClub cohorts.
    - Samples down to a manageable size for a portfolio project rather than
      processing the full 2.26M rows (adjust SAMPLE_SIZE as your machine allows).
"""

import numpy as np
import pandas as pd

RAW_PATH = r"D:\BankLoanRisk-Intelligence\Dataset\raw\accepted_2007_to_2018Q4.csv"
OUT_PATH = r"D:\BankLoanRisk-Intelligence\Dataset\loans_clean.csv"
MIN_VINTAGE_YEAR = 2015
SAMPLE_SIZE = 150_000   # set to None to keep everything after filtering
RANDOM_STATE = 42

KEEP_COLUMNS = [
    "id", "issue_d", "loan_amnt", "term", "int_rate", "installment",
    "grade", "sub_grade", "emp_title", "emp_length", "home_ownership",
    "annual_inc", "verification_status", "purpose", "addr_state", "dti",
    "delinq_2yrs", "fico_range_low", "fico_range_high", "inq_last_6mths",
    "open_acc", "revol_bal", "revol_util", "total_acc",
    "loan_status", "last_pymnt_d", "last_pymnt_amnt", "total_pymnt", "out_prncp",
]

RESOLVED_STATUSES = {"Fully Paid": 0, "Charged Off": 1, "Default": 1}


def load_raw(path: str) -> pd.DataFrame:
    # Only read the columns we need — this file is 151 columns wide, no need to
    # load it all into memory
    df = pd.read_csv(path, usecols=lambda c: c in KEEP_COLUMNS, low_memory=False)
    return df


def clean(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # --- Resolve target variable, drop unresolved loans ---
    df = df[df["loan_status"].isin(RESOLVED_STATUSES.keys())]
    df["defaulted"] = df["loan_status"].map(RESOLVED_STATUSES)

    # --- Parse issue_d into a real date + vintage_month ---
    df["issue_d"] = pd.to_datetime(df["issue_d"], format="%b-%Y", errors="coerce")
    df = df.dropna(subset=["issue_d"])
    df["vintage_month"] = df["issue_d"].dt.to_period("M").astype(str)
    df = df[df["issue_d"].dt.year >= MIN_VINTAGE_YEAR]

    # --- Parse term ("36 months" -> 36) ---
    df["term_months"] = df["term"].astype(str).str.extract(r"(\d+)").astype(float)

    # --- Derive monthly income, bureau score midpoint ---
    df["monthly_income"] = df["annual_inc"] / 12
    df["bureau_score"] = (df["fico_range_low"] + df["fico_range_high"]) / 2

    # --- Basic dedup / sanity filtering ---
    df = df.drop_duplicates(subset=["id"])
    df = df[(df["loan_amnt"] > 0) & (df["monthly_income"] > 0)]

    # --- Document known missingness rather than silently imputing everything ---
    missing_summary = df.isna().mean().sort_values(ascending=False)
    missing_summary = missing_summary[missing_summary > 0]
    if len(missing_summary):
        print("Columns with missing values (document these in data_quality_notes.md):")
        print(missing_summary.round(4))

    # Fill a small number of known-safe columns; leave the rest for the EDA
    # notebook to handle deliberately rather than guessing here
    df["emp_length"] = df["emp_length"].fillna("Unknown")
    df["revol_util"] = df["revol_util"].fillna(df["revol_util"].median())
    df["dti"] = df["dti"].fillna(df["dti"].median())

    # --- Optional downsample for a tractable portfolio-project size ---
    if SAMPLE_SIZE and len(df) > SAMPLE_SIZE:
        df = df.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE)

    return df.reset_index(drop=True)


def main():
    print("Loading raw LendingClub file (this may take a minute — it's a large CSV)...")
    raw = load_raw(RAW_PATH)
    print(f"Raw rows loaded: {len(raw):,}")

    cleaned = clean(raw)
    print(f"\nRows after filtering to resolved, {MIN_VINTAGE_YEAR}+ vintage loans: {len(cleaned):,}")
    print(f"Default rate in cleaned set: {cleaned['defaulted'].mean():.2%}")

    cleaned.to_csv(OUT_PATH, index=False)
    print(f"\nCleaned dataset written to {OUT_PATH}")
    print("Next: open Python/02_EDA_and_cleaning.ipynb and continue from this file.")


if __name__ == "__main__":
    main()


Loading raw LendingClub file (this may take a minute — it's a large CSV)...
Raw rows loaded: 2,260,701
Columns with missing values (document these in data_quality_notes.md):
emp_title         0.0654
emp_length        0.0645
last_pymnt_d      0.0022
revol_util        0.0006
dti               0.0000
inq_last_6mths    0.0000
dtype: float64

Rows after filtering to resolved, 2015+ vintage loans: 150,000
Default rate in cleaned set: 21.55%

Cleaned dataset written to D:\BankLoanRisk-Intelligence\Dataset\loans_clean.csv
Next: open Python/02_EDA_and_cleaning.ipynb and continue from this file.
